# Job market exploration with Pandas
This notebook uses the deterministic, synthetic SkillMatch seed dataset. Company names are illustrative; these are not real vacancies. Run all cells from top to bottom.


## 1. Load and inspect
Read 200 jobs and 300 skills and inspect column types.

In [1]:
from pathlib import Path
import pandas as pd
root = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
jobs = pd.read_csv(root / 'data' / 'jobs.csv')
skills = pd.read_csv(root / 'data' / 'skills.csv')
jobs.head()

,id,title,company,location,employment_type,salary_min,salary_max,description,skills,posted_date
0,1,Senior Frontend Developer,Linear,Remote,Contract,135000,175000,Join the Linear team as a senior frontend deve...,React|TypeScript|JavaScript|CSS|Next.js|Git,2026-09-01
1,2,Full Stack Engineer,Vercel,"San Francisco, CA",Full-time,90000,130000,Join the Vercel team as a full stack engineer ...,React|Node.js|TypeScript|PostgreSQL|Docker|RES...,2026-09-02
2,3,Product Designer,Notion,"New York, NY",Full-time,85000,125000,Join the Notion team as a product designer and...,Figma|UI Design|UX Research|Prototyping|Design...,2026-09-03
3,4,Python Backend Engineer,Stripe,"London, UK",Full-time,140000,180000,Join the Stripe team as a python backend engin...,Python|FastAPI|PostgreSQL|Docker|Redis|Pytest,2026-09-04
4,5,Data Scientist,Figma,"Bengaluru, IN",Full-time,105000,145000,Join the Figma team as a data scientist and he...,Python|Pandas|Machine Learning|SQL|Statistics|...,2026-09-05


## 2. Inspect quality
Count missing values and duplicate records before cleaning.

In [2]:
print(jobs.shape, skills.shape)
print(jobs.dtypes)
print(jobs.isna().sum())
print('Duplicate IDs:', jobs.duplicated(subset=['id']).sum())

(200, 10) (300, 2)
id                  int64
title              object
company            object
location           object
employment_type    object
salary_min          int64
salary_max          int64
description        object
skills             object
posted_date        object
dtype: object
id                 0
title              0
company            0
location           0
employment_type    0
salary_min         0
salary_max         0
description        0
skills             0
posted_date        0
dtype: int64
Duplicate IDs: 0


## 3. Clean missing values and duplicates
Inject a controlled missing location and duplicate row to demonstrate cleaning. Keep the original seed data intact.

In [3]:
dirty = pd.concat([jobs, jobs.iloc[[0]]], ignore_index=True)
dirty.loc[1, 'location'] = None
clean = dirty.drop_duplicates(subset=['id']).copy()
clean['location'] = clean['location'].fillna('Not specified')
for column in ['salary_min', 'salary_max']:
    clean[column] = pd.to_numeric(clean[column], errors='coerce')
    clean[column] = clean[column].fillna(clean[column].median())
clean['posted_date'] = pd.to_datetime(clean['posted_date'])
assert len(clean) == 200
assert clean['location'].isna().sum() == 0
clean.head()

,id,title,company,location,employment_type,salary_min,salary_max,description,skills,posted_date
0,1,Senior Frontend Developer,Linear,Remote,Contract,135000,175000,Join the Linear team as a senior frontend deve...,React|TypeScript|JavaScript|CSS|Next.js|Git,2026-09-01
1,2,Full Stack Engineer,Vercel,Not specified,Full-time,90000,130000,Join the Vercel team as a full stack engineer ...,React|Node.js|TypeScript|PostgreSQL|Docker|RES...,2026-09-02
2,3,Product Designer,Notion,"New York, NY",Full-time,85000,125000,Join the Notion team as a product designer and...,Figma|UI Design|UX Research|Prototyping|Design...,2026-09-03
3,4,Python Backend Engineer,Stripe,"London, UK",Full-time,140000,180000,Join the Stripe team as a python backend engin...,Python|FastAPI|PostgreSQL|Docker|Redis|Pytest,2026-09-04
4,5,Data Scientist,Figma,"Bengaluru, IN",Full-time,105000,145000,Join the Figma team as a data scientist and he...,Python|Pandas|Machine Learning|SQL|Statistics|...,2026-09-05


## 4. Filter and sort
Find remote roles with minimum salaries of at least $120,000, ordered by salary.

In [4]:
remote = clean.loc[(clean['location'] == 'Remote') & (clean['salary_min'] >= 120000)]
remote.sort_values(['salary_max', 'title'], ascending=[False, True])[['title', 'company', 'salary_min', 'salary_max']].head(15)

,title,company,salary_min,salary_max
65,Machine Learning Engineer,Supabase,155000,195000
195,Machine Learning Engineer,Clerk,155000,195000
55,Machine Learning Engineer,Clerk,150000,190000
70,Senior Frontend Developer,Loom,150000,190000
125,Machine Learning Engineer,Supabase,145000,185000
40,Senior Frontend Developer,Linear,145000,185000
170,Senior Frontend Developer,Loom,145000,185000
140,Senior Frontend Developer,Linear,140000,180000
105,Machine Learning Engineer,Supabase,135000,175000
175,Machine Learning Engineer,Clerk,135000,175000


## 5. Summarize
Group opportunities by role and expand skill lists to count demand.

In [5]:
summary = clean.groupby('title').agg(openings=('id', 'count'), average_min_salary=('salary_min', 'mean'), average_max_salary=('salary_max', 'mean')).round(0)
display(summary.sort_values('average_max_salary', ascending=False))
demand = clean.assign(skill=clean['skills'].str.split('|')).explode('skill')['skill'].value_counts()
display(demand.head(15))
display(skills.groupby('category').size().rename('skill_count'))

,openings,average_min_salary,average_max_salary
title,,,
Analytics Engineer,20,124000.0,164000.0
Machine Learning Engineer,20,120750.0,160750.0
Product Designer,20,120250.0,160250.0
Mobile Developer,20,120250.0,160250.0
Python Backend Engineer,20,120000.0,160000.0
Security Engineer,20,119750.0,159750.0
Data Scientist,20,118250.0,158250.0
Senior Frontend Developer,20,117250.0,157250.0
DevOps Engineer,20,115750.0,155750.0


skill
Python         100
Docker          80
React           40
Git             40
TypeScript      40
AWS             40
SQL             40
REST APIs       40
PostgreSQL      40
Linux           40
JavaScript      20
CSS             20
Next.js         20
Node.js         20
Prototyping     20
Name: count, dtype: int64

category
AI & ML               30
Backend               30
Cloud & DevOps        30
Data                  30
Design                30
Frontend              30
Languages             30
Product & Business    30
Professional          30
Testing & Security    30
Name: skill_count, dtype: int64

## Interpretation
These counts describe a balanced synthetic teaching dataset. Do not interpret them as evidence of actual hiring demand or salary levels.

In [6]:
print(f'{len(clean)} unique jobs, {len(skills)} catalog skills, {clean.company.nunique()} example companies')

200 unique jobs, 300 catalog skills, 20 example companies
